In [2]:
pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -U ddgs


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Built-in Tool - DuckDuckGo Search

In [6]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('cjp news on 10 oct')

print(results)

CJP Delhi News: The administration in Delhi has enforced sweeping security and transport restrictions ahead of the CJP protest in the national capital on October 10. Reacting to the measures, CJP leaders accused the government of intimidating protesters, while police cited law and order concerns and a late permission request. Delhi Jantar Mantar CJP Protest Highlights: जंतर-मंतर पर सीजेपी की तरफ से फिर रविवार की सुबह 10 बजे जंतर मंतर कूच का ऐलान किया गया है. इधर, दिल्ली मेट्रो के सभी एंट्री-एग्जिट खोल दिए ... Delhi braces for a massive CJP protest with 45 metro stations shut, trains cancelled, and 23,000 security personnel deployed. Delhi will see restrictions on retail liquor sales and cancellations of several Mail and Express trains on October 10 amid planned protests by the Cockroach Janta Party (CJP), Left groups and ... News Cities Delhi Ahead of CJP's October 10 protest, Delhi Police puts 2,800 'criminals' on radar Ahead of CJP's October 10 protest, Delhi Police puts 2,800 'crimi

In [7]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-in Tool - Shell Tool

In [9]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('whoami')

print(results)

Executing command:
 whoami
shubham



## Custom Tools

In [10]:
from langchain_core.tools import tool

In [11]:
#step 1 - create a function 

def multiply(a,b):
    """Multiply two numbers"""
    return a*b

In [13]:
#step 2 - add type hint

def multiply(a:int , b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
#step 3 - add tool decorator

@tool 
def multiply(a:int, b:int)->int:
    """Multiply two numbers"""
    return a*b 

In [15]:
result = multiply.invoke({"a":3, "b":5})

In [16]:
print(result)

15


In [17]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Method 2 - Using StructuredTool


In [19]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [21]:
class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [22]:
def multiply_func(a:int,b:int)->int:
    return a*b

In [23]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [24]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [25]:
from langchain.tools import BaseTool
from typing import Type

In [26]:
class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [28]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [29]:
multiplytool = MultiplyTool()

In [30]:
result = multiplytool.invoke({'a':3, 'b':3})

print(result)
print(multiplytool.name)
print(multiplytool.description)

print(multiplytool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [38]:
from langchain_core.tools import tool

# Custom tools
@tool
def addd(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiplyy(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [39]:
class MathToolkit:
    def get_tools(self):
        return [addd, multiplyy]


In [33]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


In [40]:
print(addd.invoke({"a": 10, "b": 5}))
print(multiplyy.invoke({"a": 10, "b": 5}))

15
50
